In [1]:
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    force=True  # Force reconfiguration even if already configured
)

import os
import sys
import pandas as pd
import numpy as np
import tensorflow as tf # type: ignore

import matplotlib.pyplot as plt
import seaborn as sns

from meridian.model import model
from meridian.model import spec
from meridian.analysis import optimizer
from meridian.analysis import analyzer

from meridian.planner.flex_budget_planner import FlexibleBudgetPlanner, CompareOptimizedVsNonOptimized
from meridian.analysis.optimizer import OptimizationResults


2025-10-07 10:20:45,474 - arviz.preview - INFO - arviz_base not installed
2025-10-07 10:20:45,475 - arviz.preview - INFO - arviz_stats not installed
2025-10-07 10:20:45,475 - arviz.preview - INFO - arviz_plots not installed


In [2]:
# optimizer input path
optimizer_input_path = "./../inputs/sample_optimizer_input_coeff.xlsx"
if not os.path.exists(optimizer_input_path):
    raise FileNotFoundError(f'File not found: {optimizer_input_path}')

# configuration
input_config = {

  # time and geo inputs
  'time_col': 'week',
  'geo_col': 'geo',
  'population_col': 'population',

  # kpi inputs
  'kpi_col': 'conversions',  #
  'kpi_type': 'non_revenue',
  'revenue_per_kpi_col': 'revenue_per_conversion',  # needed if kpi_type is non_revenue

  # impression based media inputs
  'media_cols': ['Display_impression', 'TV_impression', 'Video_impression'],
  'media_spend_cols': ['Display_spend', 'TV_spend', 'Video_spend'],
  'media_channels': ['Display', 'TV', 'Video']

}

optimization_config = {
  'fixed_budget': True,
  'use_kpi': True,

  # spend constraints
  'spend_constraint_lower': {  # (1 - value)% of historical
    'Display': 0.3,
    'TV': 0.3,
    'Video': 0.3,
  },

  'spend_constraint_upper': {  # (1 + value)% of historical
  'Display': 0.3,
  'TV': 0.3,
  'Video': 0.3,
  },

  # optimization period
  'start_date': '2024-07-06',
  'end_date': '2025-06-28'
}


In [3]:
# optimize
from fileinput import filename


planner = FlexibleBudgetPlanner(file_name=optimizer_input_path, model_config=input_config)
opt_results = planner.optimize(optimization_config)

# summarize optimized vs non-optimized
compare_opt_vs_nonopt = CompareOptimizedVsNonOptimized(opt_results)
total_opt_vs_nonopt_df = compare_opt_vs_nonopt.get_total_level_comparison()
channel_opt_vs_nonopt_df = compare_opt_vs_nonopt.get_channel_level_comparison()

# write output to a HTML file
opt_results.output_optimization_summary(filename=f'sample_optimizer_output.html', filepath=f'./../inputs/')

2025-10-07 10:20:45,620 - root - INFO - Validating config: {'time_col': 'week', 'geo_col': 'geo', 'population_col': 'population', 'kpi_col': 'conversions', 'kpi_type': 'non_revenue', 'revenue_per_kpi_col': 'revenue_per_conversion', 'media_cols': ['Display_impression', 'TV_impression', 'Video_impression'], 'media_spend_cols': ['Display_spend', 'TV_spend', 'Video_spend'], 'media_channels': ['Display', 'TV', 'Video']}
2025-10-07 10:20:45,620 - root - INFO - Optimization config validation passed
2025-10-07 10:20:45,620 - root - INFO - Building input data and inference data for optimization...
2025-10-07 10:20:45,679 - root - INFO - Detected input type: coefficients
2025-10-07 10:20:45,758 - root - INFO - Loaded Data sheet with shape: (2808, 11)
2025-10-07 10:20:45,761 - root - INFO - Loaded Parameters sheet with shape: (3, 4)
2025-10-07 10:20:45,764 - root - INFO - Loaded Coefficients sheet with shape: (36, 4)
2025-10-07 10:20:45,764 - root - INFO - Geo filtering summary:
2025-10-07 10:20:

Please report this to the TensorFlow team. When filing the bug, set the verbosity to 10 (on Linux, `export AUTOGRAPH_VERBOSITY=10`) and attach the full output.
Cause: name node, decay_function, outside of any statement?
To silence this warning, decorate the function with @tf.autograph.experimental.do_not_convert


2025-10-07 10:20:46.659814: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-10-07 10:22:13,765 - root - INFO - Budget optimization completed successfully


In [4]:
total_opt_vs_nonopt_df

,start_date,end_date,optimized_budget,optimized_total_incremental_outcome,optimized_total_cpa,nonoptimized_budget,nonoptimized_total_incremental_outcome,nonoptimized_total_cpa,budget_change,outcome_change,cpa_change
0,2024-07-06,2025-06-28,91988000.0,767692416.0,0.119824,91988000.0,758978560.0,0.1212,0.0,0.011481,-0.011351


In [5]:
channel_opt_vs_nonopt_df

,channel,optimized_spend,optimized_incremental_outcome,optimized_effectiveness,optimized_cpa,nonoptimized_spend,nonoptimized_incremental_outcome,nonoptimized_effectiveness,nonoptimized_cpa,budget_change,outcome_change,effectiveness_change,cpa_change
0,Display,10364000,111758520.0,0.031756,0.092736,14805000,127683312.0,0.025398,0.115951,-0.299966,-0.124721,0.250337,-0.200216
1,TV,50779000,429977536.0,0.220365,0.118097,52991000,439671168.0,0.215927,0.120524,-0.041743,-0.022047,0.020554,-0.020140
2,Video,30845000,225956352.0,0.062110,0.136509,24192000,191624096.0,0.067159,0.126247,0.275008,0.179165,-0.075171,0.081281


In [6]:
# response curves
opt_results.plot_response_curves()

alt.FacetChart(...)

In [7]:
# optimized vs non-optimized incremental outcome
opt_results.plot_incremental_outcome_delta()

alt.LayerChart(...)

In [8]:
# optimized spend delta
opt_results.plot_spend_delta()

alt.LayerChart(...)